In [ ]:
!pip install transformers torch pandas scikit-learn

In [ ]:
import pandas as pd
import re
import torch

from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import Trainer, TrainingArguments

In [ ]:
train_data = pd.read_csv('samsum-train.csv')
val_data = pd.read_csv('samsum-validation.csv')

print("Train Shape:", train_data.shape)
print("Validation Shape:", val_data.shape)

train_data.head()

In [ ]:
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

print("Train Shape:", train_data.shape)
print("Validation Shape:", val_data.shape)

In [ ]:
def clean_data(text):
    text = re.sub(r"\r\n", " ", str(text))
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.strip().lower()
    return text

train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

train_data['dialogue'][0]

In [ ]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

In [ ]:
def tokenize(data):
    inputs = tokenizer(
        data['dialogue'],
        padding="max_length",
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        data['summary'],
        padding="max_length",
        max_length=150,
        truncation=True
    )

    inputs['labels'] = targets['input_ids']
    return inputs   

In [ ]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

print(type(train_dataset))
print(len(train_dataset[0]['input_ids']))

In [ ]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

model.to(device)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    weight_decay=0.01,
    warmup_steps=200
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained('./saved_summery_model')
tokenizer.save_pretrained('./saved_summery_model')

In [ ]:
model = T5ForConditionalGeneration.from_pretrained('./saved_summery_model')
tokenizer = T5Tokenizer.from_pretrained('./saved_summery_model')

model.to(device)

In [ ]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)

    inputs = tokenizer(
        dialogue,
        return_tensors="pt",
        padding="max_length",
        max_length=512,
        truncation=True
    ).to(device)

    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary

In [ ]:
test_dialogue = """
A: Hey, are we still meeting today?
B: Yes, at 5 PM.
A: Great, see you then.
"""

summary = summarize_dialogue(test_dialogue)
print("Summary:", summary)